In [1]:
from dataclasses import dataclass
from typing import List, Dict, Optional

In [2]:
@dataclass
class CognitiveGoal:
    goal_id: str
    patient_id: str
    goal_type: str
    priority: int
    description: str
    source: str
    active: bool = True

In [3]:
class GoalAgent:

    def __init__(self, repository):
        self.repository = repository

    def get_active_goals(self, patient_id: str):

        goals_data = self.repository.get_patient_goals(patient_id)

        goals = []

        for data in goals_data:

            if not data.get("active", True):
                continue

            goal = CognitiveGoal(
                goal_id=data["goal_id"],
                patient_id=data["patient_id"],
                goal_type=data["goal_type"],
                priority=data["priority"],
                description=data["description"],
                source=data["source"],
                active=data.get("active", True)
            )

            goals.append(goal)

        return goals

    def select_goal(
        self,
        patient_id: str,
        current_time: str,
        current_routines=None
    ):

        goals = self.get_active_goals(patient_id)

        if not goals:
            return None

        # Highest priority first
        goals = sorted(
            goals,
            key=lambda goal: goal.priority,
            reverse=True
        )

        return goals[0]
class PatientRepository:

    def __init__(self, db):
        self.db = db

    def get_patient_goals(self, patient_id: str):

        docs = (
            self.db.collection("goals")
            .where("patient_id", "==", patient_id)
            .stream()
        )

        return [doc.to_dict() for doc in docs]

In [4]:
import firebase_admin
from firebase_admin import credentials, firestore

SERVICE_ACCOUNT_PATH = "firebase/serviceAccountKey.json.json"

if not firebase_admin._apps:
    cred = credentials.Certificate(SERVICE_ACCOUNT_PATH)
    firebase_admin.initialize_app(cred)

db = firestore.client()

In [5]:
repository = PatientRepository(db)

goal_agent = GoalAgent(repository)

print("Goal Agent connected to Firebase!")

Goal Agent connected to Firebase!


In [6]:
selected_goal = goal_agent.select_goal(
    patient_id="lakshmi_001",
    current_time="10:30",
)

print(selected_goal)

C:\Users\dmane\anaconda3\Lib\site-packages\google\cloud\firestore_v1\base_collection.py:317: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)


CognitiveGoal(goal_id='goal_001', patient_id='lakshmi_001', goal_type='routine_recall', priority=10, description='Help the patient remember daily routines.', source='caregiver', active=True)


In [7]:
def select_goal(
    self,
    patient_id: str,
    current_time: str,
    current_routines=None
):

    goals = self.get_active_goals(patient_id)

    if not goals:
        return None

    # If there is a current routine,
    # prioritize a goal related to that routine.
    if current_routines:

        routine_type = current_routines.get("routine_type")

        matching_goals = [
            goal
            for goal in goals
            if (
                routine_type == "hydration"
                and goal.goal_type == "routine_recall"
            )
        ]

        if matching_goals:
            matching_goals.sort(
                key=lambda goal: goal.priority,
                reverse=True
            )

            return matching_goals[0]

    # Fallback: highest-priority active goal
    goals.sort(
        key=lambda goal: goal.priority,
        reverse=True
    )

    return goals[0]

In [8]:
current_routine = {
    "routine_type": "hydration",
    "title": "Drink water",
    "scheduled_time": "10:30"
}

selected_goal = goal_agent.select_goal(
    patient_id="lakshmi_001",
    current_time="10:30",
    current_routines=current_routine
)

print("Selected goal:")
print(selected_goal)

Selected goal:
CognitiveGoal(goal_id='goal_001', patient_id='lakshmi_001', goal_type='routine_recall', priority=10, description='Help the patient remember daily routines.', source='caregiver', active=True)
